# train — Direction 1: SHAC through the replica, on two GPUs, packaged for submission

One GPU T4 ×2 session, GPU work only (`hvac_controller/d1/train.py`, `policy.py`, both V1's code plus one change
each); the winner's score in the organisers' simulator comes afterwards from `evaluate.ipynb`.

1. **smoke** (≈ 12 min, GPU 0), stops the notebook if anything fails: parity of the submission path (`KagglePolicy`
   on the organisers' simulator, CPU, 36 steps, on a shipped building) with the training path (GPU), a 288-step
   replica score, and a short run with the relative clip on the largest shipped building and the 4 largest
   generated ones, all 5 in every update: the most memory the `data` run's 5 buildings per update can need (peak
   below 13.5 GB, leaving room for the other buildings' tables).
2. **training**, both GPUs at once, `MINUTES` = 540 each (V2.2's guard stopped both runs at 42 and 77 of 300 min,
   still improving), V1's settings otherwise
   (h 288, γ 0.998, row budget 2.87M, Adam (0.7, 0.95), lr 2e-3 decaying over the budget):
   - `clip` (GPU 0): V1's 25 shipped training scenarios; the change is the clip, 5 × the median pre-clip norm
     of the last 50 steps instead of V1's absolute 1.0, which let spikes through at ≈ 200 × normal and froze Adam;
     it also holds per window: each building's part of the gradient is clipped to its share of that limit and
     dropped when non-finite or above 100 × it (V2.3: exploding windows no longer skip steps or stop the run).
   - `data` (GPU 1): `clip` + the generated `gen_train` scenarios (21 buildings × 3 strips, from the shipped
     buildings' ranges, those above `MAX_ZONES` left out); 5 of the 26 buildings per update, drawn at random, so each update costs what V1's did.
   - Held out (both): building_train_2's 5 shipped strips and 2 new-station strips, scored every 20 min and choosing
     the kept checkpoint (they predicted the LB of V1 and V2); at the end the kept policy is also scored on the 10
     `gen_val` scenarios (5 held-out generated buildings × 2 strips of held-out stations).
   - Gate before submitting: building_train_2 (7) below V1's 0.00551 in the replica.
3. **winner** = the run with the lower building_train_2 replica score.
4. **package:** `/kaggle/working/hvac-submission.zip` (flat: `d1_policy.pt`, `d1_policy.py`, `README.txt`), the next
   version of the `hvac-submission` dataset (`evaluate.ipynb` scores it, `submit.ipynb` submits it); and
   `/kaggle/working/train_logs_<VERSION>.zip`.

**Inputs:** the competition data and `buildings`. Accelerator **GPU T4 ×2**, Internet on. ≈ 10 h.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
VERSION = "v2.3"          # names the package's README and the logs' zip
SETTINGS = {"h": 288, "gamma": 0.998, "row_budget": 2_870_000, "clip_rel": 5.0, "eval_minutes": 20, "eval_seeds": 1}
RUNS = {"clip": {}, "data": {"generated": True, "per_update": 5}}  # run -> its additions to SETTINGS
V1_BT2 = 0.00551          # V1 on building_train_2's 7 held-out scenarios (the gate)
MINUTES = 540             # training budget of each run (≈ 10 h session; V2.2's runs were still improving)
MAX_ZONES = 110           # generated buildings above this are left out (shipped: 30-85 zones)

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch

assert torch.cuda.device_count() >= 2, "set the accelerator to GPU T4 x2"
print(torch.__version__, torch.cuda.device_count(), "x", torch.cuda.get_device_name())
REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/train")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, str(REPO))
# Expandable segments: the windows of buildings of different sizes alternate, which fragments the fixed blocks.
ENV = dict(os.environ, PYTHONPATH=str(REPO), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")


def dataset(slug):
    """One of the four datasets (docs/KAGGLE.md), wherever Kaggle mounts it."""
    return next(p for p in (INPUT / "datasets" / "cryandrrich" / slug, INPUT / slug) if p.is_dir())


SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)  # the smoke's parity
DATA = dataset("buildings")
MANIFEST = json.loads((DATA / "manifest.json").read_text())
CACHES = {p.stem: p for p in sorted(DATA.glob("*.pkl"))}
held_out = {e["building"] for e in MANIFEST if e["tag"] == "shipped_building_new_station"}  # building_train_2
assert len(held_out) == 1, held_out
shipped = sorted(f"{p.parent.parent.name}__{p.parent.name}" for p in SHIPPED.glob("*/*/config.gin"))
TRAIN_SHIPPED = [s for s in shipped if s.split("__")[0] not in held_out]
missing = [e["label"] for e in MANIFEST if e["label"] not in CACHES]
assert not missing, f"replica caches missing from buildings: {missing[:10]}"
ZONES = {s: json.loads(p.with_suffix(".json").read_text())["n_zones"] for s, p in CACHES.items()}
big = sorted({e["building"] for e in MANIFEST if ZONES[e["label"]] > MAX_ZONES})
print(f"left out (> {MAX_ZONES} zones): {big}")
GENERATED = [e["label"] for e in MANIFEST if e["tag"] == "gen_train" and e["building"] not in big]
VAL = ([s for s in shipped if s.split("__")[0] in held_out]
       + [e["label"] for e in MANIFEST if e["tag"] in ("shipped_building_new_station", "gen_val")
          and e["building"] not in big])
assert all(s in CACHES for s in TRAIN_SHIPPED + VAL)
SETTINGS["select"] = f"{next(iter(held_out))}__"  # its 7 held-out scenarios choose the kept checkpoint
print(f"{len(TRAIN_SHIPPED)} shipped + {len(GENERATED)} generated training, {len(VAL)} held-out scenarios; "
      f"{len(CACHES)} caches; select {SETTINGS['select']}")
WORK.mkdir(parents=True, exist_ok=True)  # outside /kaggle/working: the output is the two zips alone


def train_cmd(out_dir, settings, train, extra=()):
    cmd = [sys.executable, "-m", "hvac_controller.d1.train", str(out_dir),
           "--caches", ",".join(str(CACHES[s]) for s in train), "--val-caches", ",".join(str(CACHES[s]) for s in VAL)]
    for k, v in settings.items():
        cmd += [f"--{k.replace('_', '-')}", str(v)]
    return cmd + list(extra)


def largest(labels, n):
    """All scenarios of the n buildings with the most zones among labels."""
    by = {}
    for s in labels:
        by.setdefault(s.split("__")[0], []).append(s)
    return [s for b in sorted(by, key=lambda b: -ZONES[by[b][0]])[:n] for s in by[b]]


def training_set(run):
    return TRAIN_SHIPPED + (GENERATED if RUNS[run].get("generated") else [])


def tail(path, n=3000):
    text = Path(path).read_text() if Path(path).exists() else ""
    return text[-8000:] if "Traceback" in text else text[-n:]

In [ ]:
# 1. Smoke on GPU 0: parity on a shipped building, a 288-step replica score, and 7 minutes (past the 5 steps the
# relative clip waits for) of updates on the largest shipped building and the 4 largest generated ones, the most
# memory 5 buildings per update can need. Stops here if anything is wrong.
smoke = WORK / "smoke"
smoke.mkdir(exist_ok=True)
SMOKE = largest(TRAIN_SHIPPED, 1) + largest(GENERATED, 4)
t0 = time.time()
with open(smoke / "log.txt", "w") as log:
    rc = subprocess.run(train_cmd(smoke, {**SETTINGS, "minutes": 7, "eval_minutes": 1000, "eval_steps": 288}, SMOKE,
                                  ["--smoke"]),
                        cwd=REPO, env=dict(ENV, CUDA_VISIBLE_DEVICES="0"), stdout=log, stderr=subprocess.STDOUT).returncode
print(f"smoke exit {rc} in {time.time() - t0:.0f}s\n" + tail(smoke / "log.txt", 6000))
rep = json.loads((smoke / "report.json").read_text()) if (smoke / "report.json").exists() else {}
assert rc == 0 and rep.get("parity", {}).get("ok") and rep.get("iterations"), "smoke failed: read the log"
rows = [json.loads(l) for l in (smoke / "train_log.jsonl").read_text().splitlines()]
peak = max(r["mem_gb"] for r in rows)
print(f"{len(rows)} updates, {sum(r['s'] for r in rows) / len(rows):.0f} s per update of {len(rows[0]['buildings'])} "
      f"buildings, peak {peak:.1f} GB, clips {[round(r['clip'], 4) for r in rows]}")
assert "out of memory" not in (smoke / "log.txt").read_text() and peak < 13.5, "too close to the T4's 14.6 GB"


In [ ]:
# 2. The two runs, one per GPU; evaluations, collapses and bad gradients are echoed as they happen.
t0, procs = time.time(), {}
for gpu, (name, extra) in enumerate(RUNS.items()):
    d = WORK / name
    d.mkdir(exist_ok=True)
    log = open(d / "log.txt", "w")
    settings = {**SETTINGS, "minutes": MINUTES, **{k: v for k, v in extra.items() if k != "generated"}}
    procs[name] = (subprocess.Popen(train_cmd(d, settings, training_set(name)), cwd=REPO, stdout=log,
                                    stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))), log, d)
    print(f"GPU {gpu}: {name}, {len(training_set(name))} training scenarios, {settings}", flush=True)
seen = {name: 0 for name in procs}
while True:
    running = any(p.poll() is None for p, _, _ in procs.values())
    for name, (p, _, d) in procs.items():
        lines = (d / "log.txt").read_text().splitlines()
        for line in lines[seen[name]:]:
            if line.startswith(("[eval]", "Traceback")) or any(w in line for w in ("collapse", "dropped",
                                                                                     "burn-in", "out of memory")):
                print(f"[{(time.time() - t0) / 60:5.0f} min] {name}: {line[:400]}", flush=True)
        seen[name] = len(lines)
    if not running:
        break
    time.sleep(60)
REPORTS = {}
for name, (p, log, d) in procs.items():
    log.close()
    REPORTS[name] = json.loads((d / "report.json").read_text()) if (d / "report.json").exists() else {}
    kept = REPORTS[name].get("kept", {})
    gen = [v["score"] for k, v in kept.items() if isinstance(v, dict) and not k.startswith(SETTINGS["select"])]
    print(f"===== {name}: exit {p.returncode}, best on {SETTINGS['select']} {REPORTS[name].get('best')} "
          f"(V1 {V1_BT2}), gen_val {sum(gen) / max(len(gen), 1):.5f}, {REPORTS[name].get('iterations')} updates, "
          f"{REPORTS[name].get('rollbacks')} rollbacks, {REPORTS[name].get('dropped')} windows dropped, "
          f"{REPORTS[name].get('skipped')} updates without a step\n" + tail(d / "log.txt", 2500))
done = {n: r["best"] for n, r in REPORTS.items() if r.get("best") is not None and (WORK / n / "d1_policy.pt").exists()}
WINNER = min(done, key=done.get)
print("winner:", WINNER, done, "| below V1 on building_train_2:", done[WINNER] < V1_BT2)

In [ ]:
# 3. The package (the next hvac-submission version) and the logs.
readme = (f"Direction 1 {VERSION} policy ({WINNER}), commit {COMMIT}: trained on {len(training_set(WINNER))} "
          f"scenarios, chosen on {SETTINGS['select']}'s held-out ones (replica {done[WINNER]:.5f}, V1 {V1_BT2}; all "
          f"runs {json.dumps(done)}). d1_policy.py = hvac_controller/d1/policy.py. evaluate.ipynb scores it in the "
          f"organisers' simulator, submit.ipynb submits it.\n")
name = Path("/kaggle/working/hvac-submission.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(WORK / WINNER / "d1_policy.pt", "d1_policy.pt")
    z.write(REPO / "hvac_controller" / "d1" / "policy.py", "d1_policy.py")
    z.writestr("README.txt", readme)
print(readme, "wrote", name, zipfile.ZipFile(name).namelist())
name = Path(f"/kaggle/working/train_logs_{VERSION}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in WORK.rglob("*"):
        if p.is_file() and (p.suffix != ".pt" or p.name == "d1_policy.pt"):
            z.write(p, p.relative_to(WORK))
print("wrote", name)